# NB_01_A_ENGINEERING_OBJECT

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thinkthoughts/sensors-becker/blob/main/notebooks/NB_01_A_ENGINEERING_OBJECT.ipynb)

This notebook introduces the repository's next (01) engineering artifact bundle.


In [ ]:
NOTEBOOK_ID = "NB_01_A_ENGINEERING_OBJECT"
NOTEBOOK_FILENAME = f"{NOTEBOOK_ID}.ipynb"
NOTEBOOK_VERSION = "0.1.0"
ENGINEERING_STAGE = "A"
RELEASE_FILENAME = f"{NOTEBOOK_ID}.zip"

{
    "notebook_id": NOTEBOOK_ID,
    "notebook_version": NOTEBOOK_VERSION,
    "engineering_stage": ENGINEERING_STAGE,
    "release_filename": RELEASE_FILENAME,
}


## Initialize Notebook Runtime

This operation prepares the repository environment used to generate the bundle.


In [ ]:
from __future__ import annotations

import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/thinkthoughts/sensors-becker.git"
COLAB_REPOSITORY_ROOT = Path("/content/sensors-becker")


def install_colab_repository() -> Path:
    """Clone and install the repository in a fresh Colab runtime."""

    if COLAB_REPOSITORY_ROOT.exists():
        shutil.rmtree(COLAB_REPOSITORY_ROOT)

    subprocess.run(
        ["git", "clone", "--depth", "1", REPOSITORY_URL, str(COLAB_REPOSITORY_ROOT)],
        check=True,
    )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--editable",
            str(COLAB_REPOSITORY_ROOT),
        ],
        check=True,
    )

    src_dir = COLAB_REPOSITORY_ROOT / "src"
    if str(src_dir) not in sys.path:
        sys.path.insert(0, str(src_dir))

    importlib.invalidate_caches()
    os.chdir(COLAB_REPOSITORY_ROOT)
    return COLAB_REPOSITORY_ROOT


try:
    import sensors_becker
except ModuleNotFoundError:
    repository_root = install_colab_repository()
    import sensors_becker
else:
    repository_root = Path(sensors_becker.__file__).resolve().parents[2]


if not (repository_root / "pyproject.toml").exists():
    raise FileNotFoundError(
        f"Repository root does not contain pyproject.toml: {repository_root}"
    )

from sensors_becker import initialize_notebook

runtime = initialize_notebook(
    start=repository_root,
    environment=(
        "google-colab"
        if repository_root == COLAB_REPOSITORY_ROOT
        else "repository-runtime"
    ),
)
context = runtime.context

print(f"Environment: {runtime.environment}")
print(f"Package: {Path(sensors_becker.__file__).resolve()}")
print(f"Repository root: {runtime.repository_root}")


## Validate Engineering Context

This operation validates the repository engineering context before artifact generation.


In [ ]:
runtime.validate()
print("Engineering context validation: PASSED")


## Prepare Notebook Bundle

This operation prepares the portable bundle directory and reusable drawing functions for the admitted engineering concepts.


In [ ]:
from __future__ import annotations

import json
import shutil
import zipfile
from datetime import date
from hashlib import sha256
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from IPython.display import Image, Markdown, display

FOOTER = "Admissible generalizations trail leading specifications."
SUBTITLE = "Toward next-generation microcalorimeters."


def display_artifact(path: Path) -> None:
    """Display a generated artifact in the notebook and print its bundle name."""
    if not path.exists():
        raise FileNotFoundError(f"Artifact does not exist: {path}")

    suffix = path.suffix.lower()
    if suffix == ".png":
        display(Image(filename=str(path)))
    elif suffix == ".md":
        display(Markdown(path.read_text(encoding="utf-8")))
    else:
        print(path.read_text(encoding="utf-8"))

    print(f"Artifact: {path.name}")


def sha256_for_path(path: Path) -> str:
    digest = sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def draw_box(
    ax,
    x: float,
    y: float,
    width: float,
    height: float,
    label: str,
    *,
    linewidth: float = 1.8,
    fontsize: float = 17,
    fontweight: str = "semibold",
):
    patch = FancyBboxPatch(
        (x - width / 2, y - height / 2),
        width,
        height,
        boxstyle="round,pad=0.012,rounding_size=0.025",
        facecolor="white",
        edgecolor="black",
        linewidth=linewidth,
    )
    ax.add_patch(patch)
    ax.text(
        x,
        y,
        label,
        ha="center",
        va="center",
        fontsize=fontsize,
        fontweight=fontweight,
        wrap=True,
    )


def draw_arrow(ax, x1: float, y1: float, x2: float, y2: float):
    ax.annotate(
        "",
        xy=(x2, y2),
        xytext=(x1, y1),
        arrowprops={"arrowstyle": "->", "linewidth": 1.8},
    )


def draw_line(ax, x1: float, y1: float, x2: float, y2: float, *, dashed=False):
    ax.plot(
        [x1, x2],
        [y1, y2],
        linewidth=1.6,
        linestyle="--" if dashed else "-",
    )


def begin_figure(title: str):
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.text(
        0.5,
        0.94,
        title,
        ha="center",
        va="center",
        fontsize=24,
        fontweight="bold",
    )
    ax.text(
        0.5,
        0.885,
        SUBTITLE,
        ha="center",
        va="center",
        fontsize=15,
        style="italic",
    )
    return fig, ax


def finish_figure(fig, ax, output_path: Path):
    ax.text(
        0.5,
        0.04,
        FOOTER,
        ha="center",
        va="center",
        fontsize=13,
        style="italic",
    )
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)


def render_engineering_object_trail(output_path: Path) -> None:
    fig, ax = begin_figure("Engineering Object Trail: Microcalorimeters")
    ax.text(0.5, 0.77, "Engineering Object", ha="center", fontsize=18, fontweight="bold")
    draw_box(ax, 0.5, 0.64, 0.36, 0.12, "Microcalorimeter", linewidth=2.4, fontsize=20)

    ax.text(
        0.5,
        0.49,
        "Connected Engineering Structures",
        ha="center",
        fontsize=17,
        fontweight="bold",
    )
    structure_positions = [
        (0.17, "Absorber"),
        (0.39, "Thermometer"),
        (0.61, "Thermal Link"),
        (0.83, "Readout Interface"),
    ]
    for x, label in structure_positions:
        draw_box(ax, x, 0.37, 0.18, 0.09, label, fontsize=14)
        draw_line(ax, 0.5, 0.58, x, 0.415)

    ax.text(0.5, 0.22, "Engineering Constraint", ha="center", fontsize=17, fontweight="bold")
    draw_box(ax, 0.5, 0.13, 0.34, 0.08, "Cryogenic Environment", fontsize=15)
    finish_figure(fig, ax, output_path)


def render_connected_structure_trail(output_path: Path) -> None:
    fig, ax = begin_figure("Connected Structure Trail: Microcalorimeters")
    draw_box(ax, 0.5, 0.60, 0.30, 0.13, "Thermometer", linewidth=2.6, fontsize=21)

    supports = [
        (0.20, 0.68, "Absorber"),
        (0.20, 0.43, "Thermal Link"),
        (0.80, 0.55, "Readout Interface"),
    ]
    for x, y, label in supports:
        draw_box(ax, x, y, 0.22, 0.09, label, fontsize=14)
        draw_line(
            ax,
            x + (0.11 if x < 0.5 else -0.11),
            y,
            0.35 if x < 0.5 else 0.65,
            0.60,
            dashed=True,
        )

    draw_box(ax, 0.5, 0.22, 0.34, 0.09, "Cryogenic Environment", fontsize=15)
    draw_line(ax, 0.5, 0.535, 0.5, 0.265, dashed=True)
    finish_figure(fig, ax, output_path)


def render_structure_function_trail(output_path: Path) -> None:
    fig, ax = begin_figure("Structure Function Trail: Microcalorimeters")
    draw_box(ax, 0.5, 0.70, 0.30, 0.10, "Temperature", fontsize=18)
    draw_box(ax, 0.5, 0.50, 0.30, 0.11, "Thermometer", linewidth=2.5, fontsize=20)
    draw_box(ax, 0.5, 0.30, 0.30, 0.10, "Electrical Signal", fontsize=18)
    draw_arrow(ax, 0.5, 0.65, 0.5, 0.555)
    draw_arrow(ax, 0.5, 0.445, 0.5, 0.35)

    draw_box(ax, 0.18, 0.58, 0.20, 0.08, "Absorber", fontsize=14)
    draw_box(ax, 0.18, 0.40, 0.20, 0.08, "Thermal Link", fontsize=14)
    draw_line(ax, 0.28, 0.58, 0.35, 0.53, dashed=True)
    draw_line(ax, 0.28, 0.40, 0.35, 0.47, dashed=True)

    draw_box(ax, 0.80, 0.21, 0.25, 0.08, "Cryogenic Environment", fontsize=14)
    draw_line(ax, 0.68, 0.21, 0.65, 0.30, dashed=True)
    finish_figure(fig, ax, output_path)


def render_measured_state_trail(output_path: Path) -> None:
    fig, ax = begin_figure("Measured State Trail: Microcalorimeters")
    draw_box(ax, 0.5, 0.66, 0.32, 0.11, "Incident Energy", fontsize=19)
    draw_box(ax, 0.5, 0.43, 0.32, 0.12, "Measured State", linewidth=2.6, fontsize=21)
    draw_arrow(ax, 0.5, 0.605, 0.5, 0.49)

    draw_box(ax, 0.20, 0.43, 0.22, 0.09, "Thermometer", fontsize=14)
    draw_line(ax, 0.31, 0.43, 0.34, 0.43, dashed=True)

    draw_box(ax, 0.80, 0.27, 0.25, 0.09, "Cryogenic Environment", fontsize=14)
    draw_line(ax, 0.68, 0.29, 0.64, 0.39, dashed=True)
    finish_figure(fig, ax, output_path)


bundle_directory = runtime.paths.outputs / "releases" / NOTEBOOK_ID
bundle_directory.mkdir(parents=True, exist_ok=True)

for prior_path in bundle_directory.iterdir():
    if prior_path.is_file():
        prior_path.unlink()

print(f"Bundle directory: {runtime.relative_path(bundle_directory)}")


## First (A)

This notebook introduces

**01_A_engineering_object_trail.png**


In [ ]:
artifact_a_png = bundle_directory / "01_A_engineering_object_trail.png"
artifact_a_alt = bundle_directory / "01_A_engineering_object_trail.alt.md"

render_engineering_object_trail(artifact_a_png)

artifact_a_alt.write_text(
    """# Alt Text

**Engineering Object Trail: Microcalorimeters.** Diagram presenting a microcalorimeter as the engineering object, connected to an absorber, thermometer, thermal link, and readout interface. A cryogenic environment appears as the engineering constraint. Subtitle: “Toward next-generation microcalorimeters.” Footer: “Admissible generalizations trail leading specifications.”
""",
    encoding="utf-8",
)

display_artifact(artifact_a_png)
display_artifact(artifact_a_alt)


## Second (B)

This notebook introduces

**01_B_connected_structure_trail.png**


In [ ]:
artifact_b_png = bundle_directory / "01_B_connected_structure_trail.png"
artifact_b_alt = bundle_directory / "01_B_connected_structure_trail.alt.md"

render_connected_structure_trail(artifact_b_png)

artifact_b_alt.write_text(
    """# Alt Text

**Connected Structure Trail: Microcalorimeters.** Diagram highlighting the thermometer as the central connected structure within a microcalorimeter. The thermometer is linked to an absorber, thermal link, and readout interface, while a cryogenic environment is shown as the governing engineering constraint. Subtitle: “Toward next-generation microcalorimeters.” Footer: “Admissible generalizations trail leading specifications.”
""",
    encoding="utf-8",
)

display_artifact(artifact_b_png)
display_artifact(artifact_b_alt)


## Third (C)

This notebook introduces

**01_C_structure_function_trail.png**


In [ ]:
artifact_c_png = bundle_directory / "01_C_structure_function_trail.png"
artifact_c_alt = bundle_directory / "01_C_structure_function_trail.alt.md"

render_structure_function_trail(artifact_c_png)

artifact_c_alt.write_text(
    """# Alt Text

**Structure Function Trail: Microcalorimeters.** Diagram showing the functional sequence Temperature → Thermometer → Electrical Signal. The absorber and thermal link appear as supporting connected structures, while the cryogenic environment remains the engineering constraint. Subtitle: “Toward next-generation microcalorimeters.” Footer: “Admissible generalizations trail leading specifications.”
""",
    encoding="utf-8",
)

display_artifact(artifact_c_png)
display_artifact(artifact_c_alt)


## Fourth (D)

This notebook introduces

**01_D_measured_state_trail.png**


In [ ]:
artifact_d_png = bundle_directory / "01_D_measured_state_trail.png"
artifact_d_alt = bundle_directory / "01_D_measured_state_trail.alt.md"

render_measured_state_trail(artifact_d_png)

artifact_d_alt.write_text(
    """# Alt Text

**Measured State Trail: Microcalorimeters.** Diagram showing incident energy leading to a measured state. A thermometer and cryogenic environment appear as supporting engineering context. Subtitle: “Toward next-generation microcalorimeters.” Footer: “Admissible generalizations trail leading specifications.”
""",
    encoding="utf-8",
)

display_artifact(artifact_d_png)
display_artifact(artifact_d_alt)


## Record Notebook Bundle

The following records identify and verify the portable artifact bundle.


In [ ]:
artifact_e_path = bundle_directory / "01_E_README.md"
artifact_f_path = bundle_directory / "01_F_notebook_metadata.json"
artifact_g_path = bundle_directory / "01_G_manifest.json"

concept_records = [
    {
        "artifact_order": "A",
        "artifact_id": "01_A_engineering_object_trail",
        "engineering_concept": "Engineering Object",
        "realizations": [artifact_a_png, artifact_a_alt],
    },
    {
        "artifact_order": "B",
        "artifact_id": "01_B_connected_structure_trail",
        "engineering_concept": "Connected Structure",
        "realizations": [artifact_b_png, artifact_b_alt],
    },
    {
        "artifact_order": "C",
        "artifact_id": "01_C_structure_function_trail",
        "engineering_concept": "Structure Function",
        "realizations": [artifact_c_png, artifact_c_alt],
    },
    {
        "artifact_order": "D",
        "artifact_id": "01_D_measured_state_trail",
        "engineering_concept": "Measured State",
        "realizations": [artifact_d_png, artifact_d_alt],
    },
]

primary_artifact_paths = [
    realization
    for record in concept_records
    for realization in record["realizations"]
]

bundle_readme = f"""# {NOTEBOOK_ID}

This bundle contains the notebook's admitted engineering concepts in portable reading order.

## Engineering Dialogue

Engineering Object

↓

Connected Structure

↓

Structure Function

↓

Measured State

## Engineering Artifacts

""" + "\n".join(
    f"- {record['artifact_order']}: {record['artifact_id']}"
    for record in concept_records
) + f"""

## Repository

{context.repository}

## Engineering Object

Microcalorimeter

## Engineering Direction

Toward next-generation microcalorimeters.

---

*{FOOTER}*
"""

artifact_e_path.write_text(bundle_readme, encoding="utf-8")

release_metadata = {
    "artifact_type": "notebook_bundle",
    "notebook_id": NOTEBOOK_ID,
    "notebook_filename": NOTEBOOK_FILENAME,
    "notebook_version": NOTEBOOK_VERSION,
    "repository": context.repository,
    "engineering_stage": ENGINEERING_STAGE,
    "engineering_scope": "engineering_object",
    "engineering_object": "microcalorimeter",
    "runtime_environment": runtime.environment,
    "generated": date.today().isoformat(),
}

artifact_f_path.write_text(
    json.dumps(release_metadata, indent=2) + "\n",
    encoding="utf-8",
)

manifest_records = []
for concept in concept_records:
    for realization_order, path in enumerate(concept["realizations"], start=1):
        manifest_records.append(
            {
                "artifact_order": concept["artifact_order"],
                "artifact_id": concept["artifact_id"],
                "engineering_concept": concept["engineering_concept"],
                "realization_order": realization_order,
                "realization_type": (
                    "png" if path.suffix.lower() == ".png" else "alt_text"
                ),
                "filename": path.name,
                "size_bytes": path.stat().st_size,
                "sha256": sha256_for_path(path),
                "status": "verified",
            }
        )

for support_order, path in enumerate([artifact_e_path, artifact_f_path], start=1):
    manifest_records.append(
        {
            "artifact_order": path.name.split("_", 2)[1],
            "artifact_id": path.stem,
            "engineering_concept": "bundle_record",
            "realization_order": support_order,
            "realization_type": path.suffix.lower().lstrip("."),
            "filename": path.name,
            "size_bytes": path.stat().st_size,
            "sha256": sha256_for_path(path),
            "status": "verified",
        }
    )

manifest = {
    "release_filename": RELEASE_FILENAME,
    "repository": context.repository,
    "notebook_id": NOTEBOOK_ID,
    "notebook_version": NOTEBOOK_VERSION,
    "engineering_stage": ENGINEERING_STAGE,
    "generated": date.today().isoformat(),
    "engineering_dialogue": [
        "Engineering Object",
        "Connected Structure",
        "Structure Function",
        "Measured State",
    ],
    "artifacts": manifest_records,
}

artifact_g_path.write_text(
    json.dumps(manifest, indent=2) + "\n",
    encoding="utf-8",
)

for path in (artifact_e_path, artifact_f_path, artifact_g_path):
    display_artifact(path)


## Verify Notebook Bundle

This operation verifies the complete nested artifact-and-realization sequence.


In [ ]:
bundle_artifact_paths = primary_artifact_paths + [
    artifact_e_path,
    artifact_f_path,
    artifact_g_path,
]

expected_names = [
    "01_A_engineering_object_trail.png",
    "01_A_engineering_object_trail.alt.md",
    "01_B_connected_structure_trail.png",
    "01_B_connected_structure_trail.alt.md",
    "01_C_structure_function_trail.png",
    "01_C_structure_function_trail.alt.md",
    "01_D_measured_state_trail.png",
    "01_D_measured_state_trail.alt.md",
    "01_E_README.md",
    "01_F_notebook_metadata.json",
    "01_G_manifest.json",
]

actual_names = [path.name for path in bundle_artifact_paths]
assert actual_names == expected_names
assert all(path.exists() and path.stat().st_size > 0 for path in bundle_artifact_paths)

with artifact_g_path.open(encoding="utf-8") as handle:
    manifest_check = json.load(handle)

for record in manifest_check["artifacts"]:
    path = bundle_directory / record["filename"]
    assert path.exists()
    assert sha256_for_path(path) == record["sha256"]

for path in bundle_artifact_paths:
    print(f"✓ {path.name} ({path.stat().st_size} bytes)")


## Package Notebook Bundle

This operation packages the verified artifacts without changing their portable reading order.


In [ ]:
release_directory = runtime.paths.outputs / "releases"
release_directory.mkdir(parents=True, exist_ok=True)
release_path = release_directory / RELEASE_FILENAME

with zipfile.ZipFile(release_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in bundle_artifact_paths:
        archive.write(path, arcname=path.name)

with zipfile.ZipFile(release_path) as archive:
    assert archive.namelist() == expected_names

assert release_path.stat().st_size > 0

print(
    f"✓ {runtime.relative_path(release_path)} "
    f"({release_path.stat().st_size} bytes)"
)


## Download Notebook Bundle

In Google Colab, this operation downloads the ZIP. Elsewhere, it prints the saved path.


In [ ]:
if runtime.environment == "google-colab":
    from google.colab import files
    files.download(str(release_path))
else:
    print(release_path)


*Admissible generalizations trail leading specifications.*
